# ComfyUI on Google Colab

Run [ComfyUI](https://github.com/comfyanonymous/ComfyUI) on a Colab GPU. Models can live on **Google Drive** so you do not re-download 4–17 GB every session.

**Before anything:** `Runtime → Change runtime type → Hardware accelerator → GPU` (T4 is enough for SD 1.5 / SDXL. A100/L4 is better for FLUX).

---

### Two ways to work

| Path | When | Cells to run |
|---|---|---|
| **A — already have models on Drive** | You uploaded checkpoints earlier, or a previous session saved them | **1 → 2 → 3 → 5** (skip section 4) |
| **B — first time / need new models** | Empty Drive, or you want extra checkpoints | **1 → 2 → 3 → 4 → 5** |

ComfyUI itself always installs on Colab's local SSD (`/content/ComfyUI`) so git + pip stay fast. Drive is used for **models, outputs, and extra custom nodes**.

Colab deletes `/content` when the runtime dies. Drive does not.


## 0 · GPU and disk check

Run this first. If it says no CUDA, switch to a GPU runtime and re-run.


In [1]:
#@title GPU and disk
import os, shutil, subprocess, sys

print("Python", sys.version.split()[0])

try:
    import torch
    cuda = torch.cuda.is_available()
    print("CUDA:", cuda)
    if cuda:
        props = torch.cuda.get_device_properties(0)
        vram_gb = props.total_memory / (1024 ** 3)
        print(f"GPU:  {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {vram_gb:.1f} GB")
        os.environ["COMFY_VRAM_GB"] = f"{vram_gb:.1f}"
        if vram_gb < 8:
            print("Note: under 8 GB VRAM — use --lowvram and smaller models (SD 1.5).")
        elif vram_gb < 16:
            print("Note: T4-class GPU — SD 1.5 and SDXL are fine. FLUX needs --lowvram and patience.")
        else:
            print("Note: plenty of VRAM for SDXL and FLUX FP8.")
    else:
        print("\nNO GPU. Runtime → Change runtime type → T4 GPU → Save, then re-run this cell.")
except Exception as e:
    print("Could not import torch:", e)

print("\nDisk /content:")
total, used, free = shutil.disk_usage("/content")
print(f"  {free/1e9:.1f} GB free  /  {total/1e9:.1f} GB total")
if free < 20e9:
    print("  Low disk. Prefer Drive for models, or skip FLUX (17 GB).")


Python 3.13.15
CUDA: True
GPU:  Tesla T4
VRAM: 14.6 GB
Note: T4-class GPU — SD 1.5 and SDXL are fine. FLUX needs --lowvram and patience.

Disk /content:
  70.1 GB free  /  120.9 GB total


## 1 · Connect Google Drive

Mount Drive and create the folder layout ComfyUI expects.

Default root: `MyDrive/ComfyUI_gdrive/`

Put any checkpoints you already own into the matching subfolder (or run section 4 later). You can skip this cell if you only want a throwaway session — models then die with the runtime.

After it mounts, you will be asked to authorize Google Drive in a popup.


In [2]:
#@title 1 · Mount Google Drive and create folders
#@markdown Leave **MOUNT_DRIVE** checked unless you want an ephemeral session.
MOUNT_DRIVE = True  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/ComfyUI_gdrive"  #@param {type:"string"}

import json, os, shutil
from pathlib import Path

CONFIG_PATH = "/content/comfy_colab_config.json"

MODEL_SUBDIRS = [
    "checkpoints", "loras", "vae", "controlnet", "clip", "clip_vision",
    "text_encoders", "unet", "diffusion_models", "upscale_models",
    "embeddings", "hypernetworks", "photomaker", "style_models",
    "diffusers", "gligen", "vae_approx", "ipadapter", "animatediff_models",
    "animatediff_motion_lora", "configs", "model_patches", "audio_encoders",
]

def load_config():
    if os.path.exists(CONFIG_PATH):
        with open(CONFIG_PATH) as f:
            return json.load(f)
    return {}

def save_config(**kwargs):
    cfg = load_config()
    cfg.update(kwargs)
    with open(CONFIG_PATH, "w") as f:
        json.dump(cfg, f, indent=2)
    return cfg

cfg = save_config(
    drive_root=DRIVE_ROOT,
    comfy_path="/content/ComfyUI",
    drive_mounted=False,
)

if not MOUNT_DRIVE:
    print("Drive skipped. Models will live under /content/ComfyUI and vanish when Colab disconnects.")
    save_config(drive_mounted=False)
else:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    root = Path(DRIVE_ROOT)
    root.mkdir(parents=True, exist_ok=True)
    for sub in MODEL_SUBDIRS:
        (root / "models" / sub).mkdir(parents=True, exist_ok=True)
    for extra in ("output", "input", "custom_nodes", "user"):
        (root / extra).mkdir(parents=True, exist_ok=True)
    save_config(drive_mounted=True, drive_root=str(root))
    print(f"Drive ready: {root}")
    print("Drop existing .safetensors / .ckpt / .pt files into models/<type>/")
    print("\nWhat is already on Drive:")
    found = 0
    for sub in MODEL_SUBDIRS:
        folder = root / "models" / sub
        files = [p for p in folder.iterdir() if p.is_file() and not p.name.startswith(".")]
        if files:
            print(f"  {sub}/")
            for p in sorted(files, key=lambda x: x.name.lower()):
                print(f"    {p.name:60s} {p.stat().st_size/1e9:7.2f} GB")
                found += 1
    if not found:
        print("  (empty — run section 4 to download, or upload files in Drive)")
    else:
        print(f"\n{found} file(s) found. You can skip section 4 and go 2 → 3 → 5.")

    if os.path.isdir("/content/drive/MyDrive"):
        t, u, f = shutil.disk_usage("/content/drive/MyDrive")
        print(f"\nDrive quota (approx): {f/1e9:.1f} GB free / {t/1e9:.1f} GB")


Mounted at /content/drive
Drive ready: /content/drive/MyDrive/ComfyUI_gdrive
Drop existing .safetensors / .ckpt / .pt files into models/<type>/

What is already on Drive:
  checkpoints/
    DreamShaper_8_pruned.safetensors                                2.13 GB
  upscale_models/
    RealESRGAN_x2plus.pth                                           0.07 GB

2 file(s) found. You can skip section 4 and go 2 → 3 → 5.

Drive quota (approx): 66.6 GB free / 120.9 GB


## 2 · Download ComfyUI

Clones ComfyUI onto the local SSD, installs Python deps, ComfyUI-Manager, `aria2` (fast downloads) and `cloudflared` (public URL).

Re-run later sessions too — it is incremental (`git pull` if the folder exists). This does **not** download image models.


In [3]:
#@title 2 · Install ComfyUI, Manager, aria2, cloudflared
#@markdown Check **UPDATE_COMFYUI** to `git pull` an existing clone.
UPDATE_COMFYUI = True  #@param {type:"boolean"}
INSTALL_MANAGER = True  #@param {type:"boolean"}

import json, os, shutil, subprocess, sys
from pathlib import Path

CONFIG_PATH = "/content/comfy_colab_config.json"
COMFY = "/content/ComfyUI"
REPO = "https://github.com/comfyanonymous/ComfyUI.git"
MANAGER_REPO = "https://github.com/Comfy-Org/ComfyUI-Manager.git"

def load_config():
    if os.path.exists(CONFIG_PATH):
        with open(CONFIG_PATH) as f:
            return json.load(f)
    return {}

def save_config(**kwargs):
    cfg = load_config()
    cfg.update(kwargs)
    with open(CONFIG_PATH, "w") as f:
        json.dump(cfg, f, indent=2)

def run(cmd, **kw):
    print("+", " ".join(cmd) if isinstance(cmd, list) else cmd)
    return subprocess.check_call(cmd, **kw)

save_config(comfy_path=COMFY)

if not os.path.isdir(COMFY):
    print("Cloning ComfyUI…")
    run(["git", "clone", "--depth", "1", REPO, COMFY])
else:
    print("ComfyUI already present.")
    if UPDATE_COMFYUI:
        print("Updating ComfyUI…")
        subprocess.call(["git", "-C", COMFY, "pull", "--ff-only"])

os.chdir(COMFY)
print("Installing Python requirements (uses the PyTorch Colab already has)…")
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

mgr_req = os.path.join(COMFY, "manager_requirements.txt")
if os.path.isfile(mgr_req):
    run([sys.executable, "-m", "pip", "install", "-q", "-r", mgr_req])

if INSTALL_MANAGER:
    mgr = os.path.join(COMFY, "custom_nodes", "ComfyUI-Manager")
    if not os.path.isdir(mgr):
        print("Installing ComfyUI-Manager…")
        run(["git", "clone", "--depth", "1", MANAGER_REPO, mgr])
    else:
        subprocess.call(["git", "-C", mgr, "pull", "--ff-only"])
    req = os.path.join(mgr, "requirements.txt")
    if os.path.isfile(req):
        run([sys.executable, "-m", "pip", "install", "-q", "-r", req])

print("Installing aria2 (model downloader)…")
subprocess.call(["apt-get", "-y", "-qq", "update"], stdout=subprocess.DEVNULL)
subprocess.call(["apt-get", "-y", "-qq", "install", "aria2"], stdout=subprocess.DEVNULL)

if not shutil.which("cloudflared"):
    print("Installing cloudflared…")
    deb = "/tmp/cloudflared.deb"
    run([
        "wget", "-q", "-O", deb,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
    ])
    subprocess.call(["dpkg", "-i", deb])
else:
    print("cloudflared already installed.")

print("\nComfyUI is ready at", COMFY)
print("Next: section 3 (use Drive models) — even if Drive is empty, run it so outputs persist.")


Cloning ComfyUI…
+ git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
Installing Python requirements (uses the PyTorch Colab already has)…
+ /usr/bin/python3 -m pip install -q -r requirements.txt
+ /usr/bin/python3 -m pip install -q -r /content/ComfyUI/manager_requirements.txt
Installing ComfyUI-Manager…
+ git clone --depth 1 https://github.com/Comfy-Org/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
+ /usr/bin/python3 -m pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-Manager/requirements.txt
Installing aria2 (model downloader)…
Installing cloudflared…
+ wget -q -O /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb

ComfyUI is ready at /content/ComfyUI
Next: section 3 (use Drive models) — even if Drive is empty, run it so outputs persist.


## 3 · Run ComfyUI from Google Drive models

**This is the cell to use when you do not want to download anything.**

It points ComfyUI at `MyDrive/ComfyUI_gdrive/models` via `extra_model_paths.yaml`, and stores `output/` + `input/` on Drive so images survive runtime resets.

Skip section 4 after this if the model list looks right.


In [4]:
#@title 3 · Link Drive models, outputs, extra nodes
#@markdown If Drive is not mounted this cell is a no-op (ComfyUI uses local folders).
#@markdown **PERSIST_CUSTOM_NODES** also loads extra nodes from Drive (`custom_nodes/`).
PERSIST_CUSTOM_NODES = True  #@param {type:"boolean"}
PERSIST_OUTPUT = True  #@param {type:"boolean"}

import json, os, shutil
from pathlib import Path

CONFIG_PATH = "/content/comfy_colab_config.json"
COMFY = "/content/ComfyUI"

def load_config():
    if os.path.exists(CONFIG_PATH):
        with open(CONFIG_PATH) as f:
            return json.load(f)
    return {"drive_mounted": False, "drive_root": "/content/drive/MyDrive/ComfyUI_gdrive", "comfy_path": COMFY}

cfg = load_config()
drive_root = Path(cfg.get("drive_root") or "/content/drive/MyDrive/ComfyUI_gdrive")
comfy = Path(cfg.get("comfy_path") or COMFY)
mounted = bool(cfg.get("drive_mounted")) and Path("/content/drive/MyDrive").exists()

if not mounted:
    print("Google Drive is not mounted.")
    print("Run section 1 first if you want to use Drive models.")
    print("Continuing with local /content/ComfyUI/models (ephemeral).")
else:
    if not comfy.is_dir():
        raise SystemExit("ComfyUI is not installed. Run section 2 first.")

    yaml_path = comfy / "extra_model_paths.yaml"
    custom_line = "    custom_nodes: custom_nodes/\n" if PERSIST_CUSTOM_NODES else ""
    yaml_path.write_text(
        "# Generated by ComfyUI Colab notebook — Drive-backed models\n"
        "gdrive:\n"
        f"    base_path: {drive_root}/\n"
        "    is_default: true\n"
        "    checkpoints: models/checkpoints/\n"
        "    configs: models/configs/\n"
        "    loras: models/loras/\n"
        "    vae: models/vae/\n"
        "    text_encoders: |\n"
        "         models/text_encoders/\n"
        "         models/clip/\n"
        "    diffusion_models: |\n"
        "         models/unet/\n"
        "         models/diffusion_models/\n"
        "    clip_vision: models/clip_vision/\n"
        "    style_models: models/style_models/\n"
        "    embeddings: models/embeddings/\n"
        "    diffusers: models/diffusers/\n"
        "    vae_approx: models/vae_approx/\n"
        "    controlnet: |\n"
        "         models/controlnet/\n"
        "         models/t2i_adapter/\n"
        "    gligen: models/gligen/\n"
        "    upscale_models: models/upscale_models/\n"
        "    hypernetworks: models/hypernetworks/\n"
        "    photomaker: models/photomaker/\n"
        "    ipadapter: models/ipadapter/\n"
        "    animatediff_models: models/animatediff_models/\n"
        "    animatediff_motion_lora: models/animatediff_motion_lora/\n"
        "    model_patches: models/model_patches/\n"
        "    audio_encoders: models/audio_encoders/\n"
        + custom_line
    )
    print(f"Wrote {yaml_path}")
    print(f"ComfyUI will read models from {drive_root}/models")

    def replace_with_link(local: Path, target: Path):
        target.mkdir(parents=True, exist_ok=True)
        if local.is_symlink() or local.exists():
            if local.is_symlink() and os.path.realpath(local) == os.path.realpath(target):
                return
            if local.is_dir() and not local.is_symlink():
                for item in local.iterdir():
                    dest = target / item.name
                    if not dest.exists():
                        shutil.move(str(item), str(dest))
            if local.is_symlink() or local.exists():
                if local.is_dir() and not local.is_symlink():
                    shutil.rmtree(local)
                else:
                    local.unlink()
        local.symlink_to(target, target_is_directory=True)
        print(f"  {local} → {target}")

    if PERSIST_OUTPUT:
        print("Persisting input / output / user on Drive:")
        replace_with_link(comfy / "output", drive_root / "output")
        replace_with_link(comfy / "input", drive_root / "input")
        replace_with_link(comfy / "user", drive_root / "user")

    print("\nModels ComfyUI will see from Drive:")
    found = 0
    models_root = drive_root / "models"
    if models_root.exists():
        for folder in sorted(p for p in models_root.iterdir() if p.is_dir()):
            files = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in
                     {".safetensors", ".ckpt", ".pt", ".pth", ".bin", ".gguf", ".sft"}]
            if files:
                print(f"  {folder.name}/")
                for p in files:
                    print(f"    {p.name:60s} {p.stat().st_size/1e9:7.2f} GB")
                    found += 1
    if found == 0:
        print("  none yet.")
        print("  Either run section 4, or upload files to Drive:")
        print(f"    {drive_root}/models/checkpoints/")
    else:
        print(f"\n{found} model file(s) linked. Skip section 4 unless you need more.")
        print("Go to section 5 to start ComfyUI.")


Wrote /content/ComfyUI/extra_model_paths.yaml
ComfyUI will read models from /content/drive/MyDrive/ComfyUI_gdrive/models
Persisting input / output / user on Drive:
  /content/ComfyUI/output → /content/drive/MyDrive/ComfyUI_gdrive/output
  /content/ComfyUI/input → /content/drive/MyDrive/ComfyUI_gdrive/input
  /content/ComfyUI/user → /content/drive/MyDrive/ComfyUI_gdrive/user

Models ComfyUI will see from Drive:
  checkpoints/
    DreamShaper_8_pruned.safetensors                                2.13 GB
  upscale_models/
    RealESRGAN_x2plus.pth                                           0.07 GB

2 model file(s) linked. Skip section 4 unless you need more.
Go to section 5 to start ComfyUI.


## 4 · Download models

Optional. Tick what you need. Files go to **Drive** when section 1 ran, otherwise to local ComfyUI folders. Existing files are skipped (resume-safe via aria2).

Free Colab Drive is 15 GB. SD 1.5 (~4 GB) or SDXL (~7 GB) fit. FLUX Schnell FP8 is ~17 GB — needs Google One or Colab local disk, not a free empty Drive.

Gated Hugging Face repos (FLUX.1-dev, etc.) need a token: Colab **Secrets** named `HF_TOKEN`, or paste it below. Civitai: secret `CIVITAI_TOKEN`.


In [7]:
#@title 4 · Download models (skip if Drive already has them)
#@markdown ### Ready-made packs
DL_SD15 = False  #@param {type:"boolean"}
DL_DREAMSHAPER8 = False  #@param {type:"boolean"}
DL_SDXL = False  #@param {type:"boolean"}
DL_SDXL_VAE = False  #@param {type:"boolean"}
DL_SDXL_LIGHTNING_LORA = False  #@param {type:"boolean"}
DL_CONTROLNET_CANNY_SDXL = False  #@param {type:"boolean"}
DL_REALESRGAN = False  #@param {type:"boolean"}
DL_FLUX_SCHNELL_FP8 = True  #@param {type:"boolean"}
#@markdown ### Tokens (leave blank to use Colab Secrets HF_TOKEN / CIVITAI_TOKEN)
HF_TOKEN = ""  #@param {type:"string"}
CIVITAI_TOKEN = ""  #@param {type:"string"}
FORCE_REDOWNLOAD = False  #@param {type:"boolean"}

import json, os, shutil, subprocess, sys
from pathlib import Path

CONFIG_PATH = "/content/comfy_colab_config.json"
COMFY = "/content/ComfyUI"

def load_config():
    if os.path.exists(CONFIG_PATH):
        with open(CONFIG_PATH) as f:
            return json.load(f)
    return {"drive_mounted": False, "drive_root": "/content/drive/MyDrive/ComfyUI_gdrive", "comfy_path": COMFY}

cfg = load_config()
mounted = bool(cfg.get("drive_mounted")) and Path("/content/drive/MyDrive").exists()
if mounted:
    models_root = Path(cfg["drive_root"]) / "models"
    print("Download destination: Google Drive")
else:
    models_root = Path(cfg.get("comfy_path") or COMFY) / "models"
    print("Download destination: local Colab disk (ephemeral)")
print(f"  {models_root}")

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

hf = (HF_TOKEN or os.environ.get("HF_TOKEN") or secret("HF_TOKEN") or "").strip()
civit = (CIVITAI_TOKEN or os.environ.get("CIVITAI_TOKEN") or secret("CIVITAI_TOKEN") or "").strip()
if hf:
    os.environ["HF_TOKEN"] = hf
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf

CATALOG = {
    "sd15": dict(
        flag=DL_SD15, folder="checkpoints", filename="v1-5-pruned-emaonly.safetensors", size="4.3 GB",
        url="https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors",
    ),
    "dreamshaper8": dict(
        flag=DL_DREAMSHAPER8, folder="checkpoints", filename="DreamShaper_8_pruned.safetensors", size="2.0 GB",
        url="https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors",
    ),
    "sdxl": dict(
        flag=DL_SDXL, folder="checkpoints", filename="sd_xl_base_1.0.safetensors", size="6.9 GB",
        url="https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors",
    ),
    "sdxl_vae": dict(
        flag=DL_SDXL_VAE, folder="vae", filename="sdxl_vae.safetensors", size="0.3 GB",
        url="https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors",
    ),
    "lightning": dict(
        flag=DL_SDXL_LIGHTNING_LORA, folder="loras", filename="sdxl_lightning_8step_lora.safetensors", size="0.4 GB",
        url="https://huggingface.co/ByteDance/SDXL-Lightning/resolve/main/sdxl_lightning_8step_lora.safetensors",
    ),
    "canny": dict(
        flag=DL_CONTROLNET_CANNY_SDXL, folder="controlnet", filename="controlnet-canny-sdxl-1.0.fp16.safetensors", size="2.5 GB",
        url="https://huggingface.co/diffusers/controlnet-canny-sdxl-1.0/resolve/main/diffusion_pytorch_model.fp16.safetensors",
    ),
    "esrgan": dict(
        flag=DL_REALESRGAN, folder="upscale_models", filename="RealESRGAN_x2plus.pth", size="0.06 GB",
        url="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
    ),
    "flux": dict(
        flag=DL_FLUX_SCHNELL_FP8, folder="checkpoints", filename="flux1-schnell-fp8.safetensors", size="17.2 GB",
        url="https://huggingface.co/Comfy-Org/flux1-schnell/resolve/main/flux1-schnell-fp8.safetensors",
    ),
}

def dest_ok(path: Path) -> bool:
    if FORCE_REDOWNLOAD:
        return False
    return path.is_file() and path.stat().st_size > 1_000_000

def aria2_download(url: str, dest_dir: Path, filename: str):
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / filename
    if dest_ok(dest):
        print(f"SKIP  {filename}  already there ({dest.stat().st_size/1e9:.2f} GB)")
        return
    if dest.exists() and FORCE_REDOWNLOAD:
        dest.unlink()
    cmd = [
        "aria2c", "-x", "16", "-s", "16", "-k", "1M", "-c",
        "--file-allocation=none", "--console-log-level=notice", "--summary-interval=10",
        "-d", str(dest_dir), "-o", filename,
    ]
    final_url = url
    if "huggingface.co" in url and hf:
        cmd += [f"--header=Authorization: Bearer {hf}"]
    if "civitai.com" in url and civit:
        sep = "&" if "?" in url else "?"
        final_url = f"{url}{sep}token={civit}"
    cmd.append(final_url)
    print(f"GET   {filename}")
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        print("aria2 failed, trying huggingface_hub / wget…")
        if "huggingface.co" in url:
            try:
                from huggingface_hub import hf_hub_download
            except ImportError:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
                from huggingface_hub import hf_hub_download
            # https://huggingface.co/{repo}/resolve/main/{file}
            parts = url.split("huggingface.co/")[-1]
            repo, _, rest = parts.partition("/resolve/")
            rev_file = rest.split("/", 1)
            revision = rev_file[0] if len(rev_file) > 1 else "main"
            fname = rev_file[-1]
            hf_hub_download(
                repo_id=repo, filename=fname, revision=revision,
                local_dir=str(dest_dir), token=hf or True,
            )
            downloaded = dest_dir / fname
            if downloaded != dest and downloaded.exists():
                downloaded.replace(dest)
        else:
            subprocess.check_call(["wget", "-c", "-O", str(dest), final_url])
    if dest.exists():
        print(f"OK    {filename}  {dest.stat().st_size/1e9:.2f} GB")
    else:
        print(f"FAIL  {filename}")

chosen = [item for item in CATALOG.values() if item["flag"]]
if not chosen:
    print("Nothing ticked. Use the custom-URL cell below, or skip to section 5.")
else:
    print(f"Queue: {len(chosen)} file(s)\n")
    for item in chosen:
        print(f"- {item['filename']}  ({item['size']})  →  models/{item['folder']}/")
        aria2_download(item["url"], models_root / item["folder"], item["filename"])
    print("\nDone. Run section 3 again if you want a fresh inventory, then start section 5.")


Download destination: Google Drive
  /content/drive/MyDrive/ComfyUI_gdrive/models
Queue: 1 file(s)

- flux1-schnell-fp8.safetensors  (17.2 GB)  →  models/checkpoints/
SKIP  flux1-schnell-fp8.safetensors  already there (17.24 GB)

Done. Run section 3 again if you want a fresh inventory, then start section 5.


### Custom URL

Paste a Hugging Face or Civitai (or any direct) file URL. Pick the ComfyUI folder so the loader finds it.


In [ ]:
#@title Custom model URL
CUSTOM_URL = ""  #@param {type:"string"}
CUSTOM_FOLDER = "checkpoints"  #@param ["checkpoints", "loras", "vae", "controlnet", "clip", "clip_vision", "text_encoders", "unet", "diffusion_models", "upscale_models", "embeddings", "ipadapter", "animatediff_models", "photomaker"]
CUSTOM_FILENAME = ""  #@param {type:"string"}
#@markdown Leave filename empty to take it from the URL. Direct file links only (…/resolve/main/file.safetensors or Civitai `api/download/models/ID`).

import json, os, subprocess
from pathlib import Path
from urllib.parse import unquote, urlparse

CONFIG_PATH = "/content/comfy_colab_config.json"

def load_config():
    if os.path.exists(CONFIG_PATH):
        with open(CONFIG_PATH) as f:
            return json.load(f)
    return {"drive_mounted": False, "drive_root": "/content/drive/MyDrive/ComfyUI_gdrive", "comfy_path": "/content/ComfyUI"}

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

cfg = load_config()
mounted = bool(cfg.get("drive_mounted")) and Path("/content/drive/MyDrive").exists()
models_root = Path(cfg["drive_root"]) / "models" if mounted else Path(cfg.get("comfy_path") or "/content/ComfyUI") / "models"
hf = (os.environ.get("HF_TOKEN") or secret("HF_TOKEN") or "").strip()
civit = (os.environ.get("CIVITAI_TOKEN") or secret("CIVITAI_TOKEN") or "").strip()

url = CUSTOM_URL.strip()
if not url:
    print("Paste a URL first.")
else:
    name = CUSTOM_FILENAME.strip()
    if not name:
        name = unquote(urlparse(url).path.rstrip("/").split("/")[-1])
        if "?" in name:
            name = name.split("?")[0]
        if not name or name in {"main", "resolve", "models"}:
            name = "downloaded_model.safetensors"
    dest_dir = models_root / CUSTOM_FOLDER
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / name
    if dest.exists() and dest.stat().st_size > 1_000_000:
        print(f"Already present: {dest} ({dest.stat().st_size/1e9:.2f} GB)")
    else:
        cmd = [
            "aria2c", "-x", "16", "-s", "16", "-k", "1M", "-c",
            "--file-allocation=none", "--console-log-level=notice",
            "-d", str(dest_dir), "-o", name,
        ]
        final = url
        if "huggingface.co" in url and hf:
            cmd += [f"--header=Authorization: Bearer {hf}"]
        if "civitai.com" in url and civit:
            sep = "&" if "?" in url else "?"
            final = f"{url}{sep}token={civit}"
        cmd.append(final)
        print(f"GET {name} → {dest_dir}")
        subprocess.check_call(cmd)
        print(f"OK  {dest}  {dest.stat().st_size/1e9:.2f} GB" if dest.exists() else "Download failed")


## 5 · Run ComfyUI

Starts the server and prints a `https://….trycloudflare.com` link. Open that in a new tab (not the Colab output iframe). Keep this cell running — stopping it kills the UI.

First boot can take 1–2 minutes while custom nodes import. If the tunnel URL does not appear, re-run the cell.

Outputs save to Drive `ComfyUI_gdrive/output/` when section 3 ran.


In [10]:
#@title 5 · Launch ComfyUI (Cloudflare tunnel)
VRAM_MODE = "auto"  #@param ["auto", "highvram", "default", "lowvram", "novram"]
PREVIEW_METHOD = "auto"  #@param ["auto", "latent2rgb", "taesd", "none"]
PORT = 8188  #@param {type:"integer"}
EXTRA_ARGS = "--enable-cors-header *"  #@param {type:"string"}

import json, os, shutil, socket, subprocess, sys, threading, time
from pathlib import Path

CONFIG_PATH = "/content/comfy_colab_config.json"
COMFY = "/content/ComfyUI"

def load_config():
    if os.path.exists(CONFIG_PATH):
        with open(CONFIG_PATH) as f:
            return json.load(f)
    return {"comfy_path": COMFY, "drive_mounted": False}

cfg = load_config()
comfy = cfg.get("comfy_path") or COMFY
if not os.path.isdir(comfy):
    raise SystemExit("ComfyUI is not installed. Run section 2 first.")

os.chdir(comfy)
os.environ["PYTHONUNBUFFERED"] = "1"

try:
    import torch
    cuda = torch.cuda.is_available()
    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if cuda else 0
except Exception:
    cuda, vram = False, 0

if not cuda:
    print("WARNING: no GPU. Switch runtime to T4 GPU.")

mode = VRAM_MODE
if mode == "auto":
    mode = "highvram" if vram >= 32 else ("default" if vram >= 12 else "lowvram")
if mode in ("normalvram", "default", "none", ""):
    vram_flag = None
else:
    vram_flag = f"--{mode}"

print(f"GPU VRAM ~{vram:.1f} GB  →  {vram_flag or 'default (no VRAM flag)'}")

subprocess.call(["pkill", "-f", "main.py --listen"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

if not shutil.which("cloudflared"):
    deb = "/tmp/cloudflared.deb"
    subprocess.check_call(["wget", "-q", "-O", deb, "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"])
    subprocess.call(["dpkg", "-i", deb])

def wait_then_tunnel(port: int):
    deadline = time.time() + 180
    while time.time() < deadline:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        try:
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                break
        finally:
            sock.close()
        time.sleep(0.5)
    else:
        print("Timed out waiting for port", port, flush=True)
        return
    print("\nComfyUI is listening. Opening Cloudflare tunnel…\n", flush=True)
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        if "trycloudflare.com" in line and "http" in line:
            url = line[line.find("http"):].strip()
            print("=" * 72, flush=True)
            print("  Open this URL in a new browser tab:", flush=True)
            print(" ", url, flush=True)
            print("=" * 72, flush=True)

threading.Thread(target=wait_then_tunnel, args=(int(PORT),), daemon=True).start()

args = [sys.executable, "-u", "main.py", "--listen", "0.0.0.0", "--port", str(int(PORT)), "--preview-method", PREVIEW_METHOD]
if vram_flag:
    args.append(vram_flag)
args.append("--use-pytorch-cross-attention")
args.extend(EXTRA_ARGS.strip().split())

print("Starting:", " ".join(args), "\n", flush=True)
code = subprocess.call(args)
print("\nComfyUI exited with code", code, flush=True)

GPU VRAM ~14.6 GB  →  default (no VRAM flag)
Starting: /usr/bin/python3 -u main.py --listen 0.0.0.0 --port 8188 --preview-method auto --use-pytorch-cross-attention --enable-cors-header * 


ComfyUI is listening. Opening Cloudflare tunnel…

  Open this URL in a new browser tab:
  https://glasses-allergy-colleges-mirrors.trycloudflare.com                                |


KeyboardInterrupt: 

## Optional · extra custom nodes

ComfyUI-Manager is already installed (section 2). Use it from the UI, or clone extra repos here. If you enabled **PERSIST_CUSTOM_NODES** in section 3, clones go onto Drive and survive the next runtime.


In [ ]:
#@title Optional · install extra custom nodes
INSTALL_CONTROLNET_AUX = False  #@param {type:"boolean"}
INSTALL_IMPACT_PACK = False  #@param {type:"boolean"}
INSTALL_IPADAPTER = False  #@param {type:"boolean"}
INSTALL_VIDEOHELPER = False  #@param {type:"boolean"}
EXTRA_GIT_URL = ""  #@param {type:"string"}

import json, os, subprocess, sys
from pathlib import Path

CONFIG_PATH = "/content/comfy_colab_config.json"

def load_config():
    if os.path.exists(CONFIG_PATH):
        with open(CONFIG_PATH) as f:
            return json.load(f)
    return {"comfy_path": "/content/ComfyUI", "drive_mounted": False, "drive_root": "/content/drive/MyDrive/ComfyUI_gdrive"}

cfg = load_config()
comfy = Path(cfg.get("comfy_path") or "/content/ComfyUI")
drive_nodes = Path(cfg.get("drive_root") or "/content/drive/MyDrive/ComfyUI_gdrive") / "custom_nodes"
yaml = comfy / "extra_model_paths.yaml"
use_drive = (
    bool(cfg.get("drive_mounted"))
    and Path("/content/drive/MyDrive").exists()
    and yaml.exists()
    and "custom_nodes" in yaml.read_text()
)
target = drive_nodes if use_drive else (comfy / "custom_nodes")
target.mkdir(parents=True, exist_ok=True)
print("Installing into", target)

PACKS = []
if INSTALL_CONTROLNET_AUX:
    PACKS.append(("comfyui_controlnet_aux", "https://github.com/Fannovel16/comfyui_controlnet_aux.git"))
if INSTALL_IMPACT_PACK:
    PACKS.append(("ComfyUI-Impact-Pack", "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git"))
if INSTALL_IPADAPTER:
    PACKS.append(("ComfyUI_IPAdapter_plus", "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git"))
if INSTALL_VIDEOHELPER:
    PACKS.append(("ComfyUI-VideoHelperSuite", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"))
url = EXTRA_GIT_URL.strip()
if url:
    name = url.rstrip("/").split("/")[-1].replace(".git", "")
    PACKS.append((name, url))

if not PACKS:
    print("Nothing selected.")
else:
    for name, repo in PACKS:
        dest = target / name
        if dest.exists():
            print(f"pull {name}")
            subprocess.call(["git", "-C", str(dest), "pull", "--ff-only"])
        else:
            print(f"clone {name}")
            subprocess.check_call(["git", "clone", "--depth", "1", repo, str(dest)])
        req = dest / "requirements.txt"
        if req.is_file():
            subprocess.call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
    print("Restart section 5 so ComfyUI picks up the new nodes.")


## Notes

- **Returning tomorrow:** GPU check → 1 Drive → 2 ComfyUI → 3 link Drive → 5 run. Skip downloads.
- **Upload your own models:** in Google Drive, put files under `ComfyUI_gdrive/models/checkpoints/` (or `loras/`, `vae/`, …), then run section 3.
- **Disconnects:** Colab kills idle GPUs. Re-run from section 1. Models on Drive are still there.
- **T4 / 15 GB Drive:** prefer DreamShaper 8 or SD 1.5. Skip FLUX on free Drive.
- **FLUX.1-dev** is gated — accept the license on Hugging Face and set `HF_TOKEN`.
- Tunnel URL changes every launch. Use the new `trycloudflare.com` link each time.
- To stop: `Runtime → Interrupt execution`, or close the tab.

Drive layout created by section 1:

```
MyDrive/ComfyUI_gdrive/
  models/checkpoints|loras|vae|controlnet|clip|text_encoders|unet|diffusion_models|upscale_models|…
  output/            (your images)
  input/
  custom_nodes/      (optional extras)
```
